# 14. Class-specific thresholds: validation ablation

Thresholds для заранее выбранных проблемных классов `BIN` и `ACT` подбираются только на validation методом coordinate descent по итоговому document-level micro-F1. Остальные классы используют глобальный threshold. Это отдельная ablation; основной результат не перезаписывается.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import runpy

PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
EXPERIMENT_CONFIG = PROJECT_DIR / 'configs/experiments/span_ner_corrected_v1.yaml'
OUTPUT_DIR = PROJECT_DIR / 'results/span_ner_corrected_v1/seed_42'
CHECKPOINT = OUTPUT_DIR / 'checkpoints/best'
GLOBAL_CALIBRATION = OUTPUT_DIR / 'threshold_calibration.json'
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'

for required_path in (EXPERIMENT_CONFIG, CHECKPOINT, GLOBAL_CALIBRATION, BOOTSTRAP):
    if not required_path.exists():
        raise FileNotFoundError(f'Не найден {required_path}. Выполните ноутбуки 10 и 11.')
global_report = json.loads(GLOBAL_CALIBRATION.read_text(encoding='utf-8'))
GLOBAL_THRESHOLD = float(global_report['best_threshold'])
bootstrap_project = runpy.run_path(str(BOOTSTRAP))['bootstrap_project']
bootstrap_project(PROJECT_DIR)

In [ ]:
from rurebus_ie.training import calibrate_span_class_thresholds_experiment

# 0.40..0.96 с шагом 0.02; значение 0.82 входит в сетку.
CLASS_THRESHOLD_GRID = [round(0.40 + step * 0.02, 2) for step in range(29)]
calibration = calibrate_span_class_thresholds_experiment(
    EXPERIMENT_CONFIG,
    thresholds=CLASS_THRESHOLD_GRID,
    initial_threshold=GLOBAL_THRESHOLD,
    max_rounds=2,
    entity_types=['BIN', 'ACT'],
    project_root=PROJECT_DIR,
    checkpoint_dir=CHECKPOINT,
)
print('Class thresholds:', calibration.class_thresholds)

In [ ]:
import pandas as pd

global_metrics = global_report['best_metrics']
comparison = pd.DataFrame(
    {
        'global': {key: global_metrics[key] for key in ('precision', 'recall', 'micro_f1', 'macro_f1')},
        'class_specific': {
            'precision': calibration.best_metrics.precision,
            'recall': calibration.best_metrics.recall,
            'micro_f1': calibration.best_metrics.micro_f1,
            'macro_f1': calibration.best_metrics.macro_f1,
        },
    }
).T
display(comparison)
display(pd.Series(calibration.class_thresholds, name='threshold').to_frame())
display(pd.DataFrame(calibration.best_metrics.per_class).T.sort_values('f1'))

In [ ]:
trace = pd.DataFrame(calibration.trace)
display(trace)
print('Отчёт:', OUTPUT_DIR / 'class_threshold_calibration.json')
print('Следующий шаг: 15_test_span_class_thresholds.ipynb')